# Modül 1b — Geometri, Morfoloji, Bitwise, HSV — problem sırasıyla (Ex 26–35)

**Durum:** 0/10. Modül 1a'nın devamı: problem numaraları 9'dan devam ediyor, yeni araçlar aynı `cvtools.py` dosyasına ekleniyor.

**Ön koşul:** Modül 1a'nın araçları (en azından P1, P2, P6, P8) yazılmış ve testleri `✓` olmalı. Bu modülün testleri onları kullanıyor.

**Oturum kuralı (1a ile aynı):** 1 oturum = 1 problem. Önce egzersiz hücreleri ve gözlemler, sonra aracı `cvtools.py`'ye yaz, test hücresini çalıştır, `PROGRESS.md`'ye tek satır.

**İsimlendirme:** `img_bgr` OpenCV için, `img_rgb` sadece Matplotlib'de göstermek için. Göstermek için artık `show()` var.

Kaynak: Week 2 OpenCV + Kaggle workbook, Part 11–14.

## `cvtools` bu modülde ne işe yarıyor?

1a'da her problemin sonunda bir araç yazdın. Bu modülde ilk kez o araçları **yeniden yazmadan kullanıyorsun**:

- Morfoloji için temizlenecek bir maske lazım → `binarize()` (P6) hazır.
- Temizliğin işe yarayıp yaramadığını saymak lazım → `find_boxes()` (P8) hazır.
- HSV maskesi lazım → `color_mask()` (P2) hazır.
- Final v2, 1a'daki cephe zincirinin aynısı, sadece iki yeni halka ekleniyor.

Araç bir kez yazılıyor, sonra her notebook'ta, her projede tek satırla çağrılıyor. Arzu yolları projesi de aynı dosyayı kullanacak.

## Problem zinciri

```text
(1a: P1–P8, final v1)
P9   Aynı şey farklı açıda, boyutta görünürse?         → rotate, resize_scale
P10  Sadece işime yarayan parçaları nasıl keserim?     → crop_boxes
P11  Kirli bir maskeyi nasıl temizlerim?               → clean_mask
P12  İki maskeyi nasıl birleştiririm?                  → combine
         │
         ▼
FINAL v2  Cephe pencereleri: v1 + clean_mask + combine + crop_boxes
          → outputs/windows/ klasöründe pencere kesitleri
```

| Araç | Final v2 | Sonrası |
|---|---|---|
| `rotate`, `resize_scale` | eğik çekilmiş cepheyi hafifçe düzeltmek, büyük fotoğrafı küçültmek | CNN eğitiminde veri çoğaltma (çevirme, döndürme); farklı ölçekli hava fotoğrafları |
| `crop_boxes` | her pencere adayını ayrı bir görüntü olarak kaydetmek | ImageToRender: eleman kesitlerini sınıflandırıcıya vermek; kendi küçük veri setin |
| `clean_mask` | doğramayla bölünmüş camı birleştirmek, tek tük gürültüyü atmak | arzu yolları: kopuk patika parçalarını birleştirmek |
| `combine` | "koyu ama gökyüzü değil" | arzu yolları: "yeşil olmayan ama bina da olmayan" |

In [1]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

IMAGE_PATH = Path("images/test.jpg")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

img_bgr = cv2.imread(str(IMAGE_PATH))
assert img_bgr is not None, f"Görsel okunamadı: {IMAGE_PATH}"
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
print(img_bgr.shape, img_bgr.dtype)

(554, 554, 3) uint8


# Problem 9 — Aynı şey farklı konumda, açıda ya da boyutta görünürse ne olur?

**Soru:** Bir cepheyi biraz eğik, biraz uzaktan, aynadan yansımış gibi çekersem piksellere ne oluyor? Görüntüyü döndürünce köşeleri neden kaybediyorum, küçültünce ne kayboluyor?

**Çıktı (araç):** `rotate(img, angle, expand=True)`, `resize_scale(img, s)`

**Neden lazım:** Gerçek fotoğraflar hiçbir zaman tam önden ve aynı ölçekte gelmiyor. Derin öğrenmede de bu dönüşümler bilerek kullanılıyor: bir modeli çevrilmiş ve döndürülmüş kopyalarla eğitmek (veri çoğaltma) onu yöne ve ölçeğe karşı daha sağlam yapıyor.

### 9.1 (Ex 26) — Kaydırma
Görüntüyü yatay ve dikey kaydır, orijinalle karşılaştır. Kaydırma matrisini kavramsal olarak yaz: `[[1, 0, tx], [0, 1, ty]]` neyi söylüyor?

In [2]:
# TODO


**Gözlem 9.1 (Ex 26)** — Matrisin hangi elemanı neyi kaydırıyor; görüntünün dışına taşan kısma ne oluyor?

> ...

### 9.2 (Ex 27) — Döndürme
90°, 180° ve kendi seçtiğin bir açı (örn. 30°). Çıktı boyutlarını karşılaştır.

In [3]:
# TODO


**Gözlem 9.2 (Ex 27)** — Özel açıda köşeler neden kesiliyor? 90° ile 30° arasındaki fark ne?

> ...

### 9.3 (Ex 28) — Yeniden boyutlandırma
%25, %50, %100, %150. Hedef boyutları elle yazma, orijinal boyuttan hesapla.

In [4]:
# TODO


**Gözlem 9.3 (Ex 28)** — Küçültünce hangi ayrıntılar kayboluyor, büyütünce görüntü neden yeni ayrıntı kazanmıyor?

> ...

### 9.4 (Ex 29) — Çevirme
Yatay, dikey, istersen ikisi birden.

In [5]:
# TODO


**Gözlem 9.4 (Ex 29)** — Her birinde hangi eksen tersine dönüyor? Bir cephe fotoğrafını yatay çevirmek bir eğitim verisi için anlamlı mı, dikey çevirmek?

> ...

### Araç 9 — `rotate`, `resize_scale`
- `rotate(img, angle, expand=True)`: görüntüyü merkezinden döndürür. `expand=True` ise tuvali büyütür ki hiçbir köşe kesilmesin: yeni genişlik `h·|sin| + w·|cos|`, yeni yükseklik `h·|cos| + w·|sin|` (yuvarlanmış). Döndürme matrisinin öteleme kısmını yeni merkeze göre kaydırman gerekecek. `expand=False` ise boyut aynı kalır.
- `resize_scale(img, s)`: boyutları `s` ile çarpar (yuvarlanmış). Küçültürken `INTER_AREA`, büyütürken `INTER_LINEAR` kullanır. `s <= 0` ise `ValueError`.

In [6]:
import importlib, cvtools; importlib.reload(cvtools)
from cvtools import rotate, resize_scale, load_bgr, show

white = np.full((100, 200), 255, np.uint8)            # 20 000 beyaz piksel
assert rotate(white, 90).shape == (200, 100), "90°'de genişlik ve yükseklik yer değiştirmeli"
assert rotate(white, 45).shape == (212, 212)
kept = int((rotate(white, 45) > 127).sum())
assert abs(kept - 20000) < 600, f"expand=True içeriği korumalı (≈20000), gelen {kept}"

cut = rotate(white, 45, expand=False)
assert cut.shape == (100, 200), "expand=False boyutu değiştirmemeli"
print("45° expand=False ile kaybolan oran:", round(1 - int((cut > 127).sum()) / 20000, 2))

img = np.zeros((100, 200, 3), np.uint8)
assert resize_scale(img, 0.25).shape == (25, 50, 3)
assert resize_scale(img, 1.5).shape == (150, 300, 3)
try:
    resize_scale(img, 0)
    raise AssertionError("s = 0 için ValueError bekleniyordu")
except ValueError:
    pass

duck = load_bgr("images/test.jpg")
show(duck, rotate(duck, 30), rotate(duck, 30, expand=False), titles=["girdi", "30° expand", "30° kesik"])
print("Problem 9 ✓")

NotImplementedError: 

# Problem 10 — Görüntünün sadece işime yarayan parçalarını nasıl keserim?

**Soru:** `find_boxes` bana kutular veriyor. Her kutunun içini ayrı bir görüntü olarak nasıl alırım; kutu görüntünün kenarına dayanıyorsa ne olur?

**Çıktı (araç):** `crop_boxes(bgr, boxes, pad=0)`

**Neden lazım:** Klasik CV ile derin öğrenme arasındaki köprü bu. Bugün kutuları sayıyorsun; ileride her kutunun içini bir sınıflandırıcıya verip "pencere mi, kapı mı, balkon mu?" diye soracaksın. ImageToRender'daki CNN yama sınıflandırıcısının girdisi tam olarak bu tür kesitler.

### 10.1 (Ex 30) — Kırpma
NumPy dilimleme ile anlamlı bir bölgeyi kes ve kaydet. Sıra: `image[y1:y2, x1:x2]`, önce satır (y), sonra sütun (x).

In [ ]:
# TODO


**Gözlem 10.1 (Ex 30)** — Kesit orijinalin kopyası mı, görünümü (view) mü? Kesitte bir pikseli değiştirince orijinale ne oluyor?

> ...

### Araç 10 — `crop_boxes`
`crop_boxes(bgr, boxes, pad=0)`: her `(x, y, w, h, ...)` kutusu için, her yandan `pad` piksel genişletilmiş bir kesit döndürür. Kesit görüntünün dışına taşmaz (sınırlarda kırpılır). Her kesit bir **kopya** olmalı: kesiti değiştirmek orijinali değiştirmemeli (Gözlem 10.1'in cevabı bunun nedeni).

In [ ]:
import importlib, cvtools; importlib.reload(cvtools)
from cvtools import crop_boxes

img = np.zeros((50, 50, 3), np.uint8)
boxes = [(10, 10, 5, 5, 25.0), (45, 45, 10, 10, 100.0)]   # ikincisi sınırın dışına taşıyor
patches = crop_boxes(img, boxes, pad=2)
assert len(patches) == 2
assert patches[0].shape == (9, 9, 3), f"5 + 2·2 = 9 bekleniyordu, gelen {patches[0].shape}"
assert patches[1].shape == (7, 7, 3), f"sınırda kırpılmalı (43..50), gelen {patches[1].shape}"

patches[0][:] = 255
assert img.sum() == 0, "kesiti değiştirmek orijinali değiştirmemeli (.copy())"
print("Problem 10 ✓")

# Problem 11 — Kirli bir maskeyi nasıl temizlerim?

**Soru:** `binarize` sonrası maskede tek tük beyaz noktalar, nesnelerin içinde delikler, ince bir çizgiyle ikiye bölünmüş parçalar var. Şekillerin ana hatlarını bozmadan bunları nasıl düzeltirim?

**Çıktı (araç):** `clean_mask(mask, open_k=3, close_k=5)`

**Neden lazım:** Final v1'de büyük ihtimalle tam bu sorunları gördün: doğramanın ikiye böldüğü cam, sıvadaki lekelerden çıkan minik kutular. Bu araç v1 ile v2 arasındaki farkın ana kaynağı. Arzu yollarında da kopuk patika parçalarını birleştirecek.

| İşlem | Ne yapar |
|---|---|
| erosion | ön planı inceltir; küçük parçalar yok olur |
| dilation | ön planı kalınlaştırır; küçük boşluklar dolar |
| opening = erosion → dilation | küçük, yalnız parçaları siler; büyükleri aşağı yukarı korur |
| closing = dilation → erosion | küçük boşlukları ve delikleri kapatır; büyükleri aşağı yukarı korur |

### 11.1 (Ex 31) — Erosion ve dilation
Aynı ikili maskeye ikisini de uygula. Maske için 1a aracını kullan:

```python
from cvtools import binarize
mask = binarize(img_gray, "otsu")
```

Karşılaştır: nesne boyutu, küçük gürültü, boşluklar.

In [ ]:
# TODO


**Gözlem 11.1 (Ex 31)** — Erosion hangi şeyleri yok etti, dilation hangi boşlukları kapattı? Kernel büyüdükçe ne değişiyor?

> ...

### 11.2 (Ex 32) — Opening ve closing
İkisini de uygula.

In [ ]:
# TODO


**Gözlem 11.2 (Ex 32)** — Hangisi küçük, yalnız bölgeleri siliyor; hangisi küçük boşlukları dolduruyor? İkisini birlikte uygularken sıra neden önemli?

> ...

### Araç 11 — `clean_mask`
`clean_mask(mask, open_k=3, close_k=5)`: önce `open_k × open_k` kare kernel ile opening, sonra `close_k × close_k` ile closing. İkisinden biri 0 ise o adım atlanır.

Test, 1a'daki `find_boxes`'ı sayaç olarak kullanıyor: kirli maskede 4 parça var (ikiye bölünmüş bir dikdörtgen ve iki gürültü noktası). Temizlikten sonra tek bir dikdörtgen kalmalı.

**Final için not:** Pencereler dikdörtgen olduğu için kare kernel iyi bir başlangıç. Cam dikey doğramayla bölünüyorsa yatay bir kernel (örn. `15 × 1`) parçaları daha seçici birleştirir; bu, final v2'de vereceğin kararlardan biri.

In [ ]:
import importlib, cvtools; importlib.reload(cvtools)
from cvtools import clean_mask, find_boxes, binarize, show

m = np.zeros((100, 100), np.uint8)
m[20:60, 10:40] = 255
m[20:60, 25] = 0          # dikdörtgeni ikiye bölen 1 piksellik çizgi
m[80, 80] = 255           # gürültü
m[5, 90] = 255            # gürültü
assert len(find_boxes(m, 0)) == 4, "kirli maskede 4 parça olmalı"

c = clean_mask(m)
assert c.dtype == np.uint8 and set(np.unique(c)) <= {0, 255}
boxes = find_boxes(c, 0)
assert len(boxes) == 1, f"temizlikten sonra tek parça bekleniyordu, gelen {len(boxes)}"
assert boxes[0][:4] == (10, 20, 30, 40), f"dikdörtgenin dış sınırı korunmalı, gelen {boxes[0][:4]}"
assert (clean_mask(m, 0, 0) == m).all(), "open_k = close_k = 0 ise maske değişmemeli"

mask = binarize(img_gray, "otsu")
show(mask, clean_mask(mask), titles=["otsu", "clean_mask"])
print("Problem 11 ✓")

# Problem 12 — İki maskeyi nasıl birleştiririm?

**Soru:** Bir maske "koyu olan yerler", diğeri "gökyüzü" diyor. "Koyu ama gökyüzü olmayan" yerleri nasıl elde ederim? Hangi soruya hangi işlem cevap veriyor?

**Çıktı (araç):** `combine(a, b, op)`

**Neden lazım:** Gerçek bir pipeline'da tek bir maske nadiren yeterli. Farklı kaynaklardan (parlaklık, renk, kenar) gelen maskeleri mantıkla birleştirmek, her birinin tek başına yaptığı hataları azaltıyor.

| İşlem | Mantık | Soru |
|---|---|---|
| `and` | A ve B | ikisinin de seçtiği yer |
| `or` | A veya B | en az birinin seçtiği yer |
| `xor` | A ya da B, ikisi birden değil | iki maskenin anlaşmadığı yer |
| `minus` | A ve değil-B | A'nın seçtiği ama B'nin dışladığı yer |

### 12.1 (Ex 33) — Bitwise maske işlemleri
İki basit maske oluştur (örn. üst üste binen bir daire ve bir dikdörtgen). `bitwise_and`, `bitwise_or`, `bitwise_not`, `bitwise_xor` sonuçlarını göster.

In [ ]:
# TODO


**Gözlem 12.1 (Ex 33)** — Her işlemin görsel anlamı ne? "minus" işlemini bu dört işlemden hangileriyle kurarsın?

> ...

### 12.2 (Ex 34) — BGR → HSV
HSV'ye çevir; `shape`, `dtype` ve bir pikseli incele. (Bunu Ex 6'da tıklama aracıyla zaten yaptın; burada kısa bir tekrar.)

In [ ]:
# TODO


**Gözlem 12.2 (Ex 34)** — Hue, Saturation ve Value kabaca neyi temsil ediyor? OpenCV'de Hue neden 0–179 aralığında?

> ...

### 12.3 (Ex 35) — Renge göre maske
Basit bir renk aralığını HSV ile ayır; orijinal, maske ve sonucu göster. Kod yazmadan önce: 1a'da bunun aracını zaten yazdın. Bu hücrede yalnızca `color_mask`, `apply_mask` ve `show` kullan.

In [ ]:
# TODO


**Gözlem 12.3 (Ex 35)** — Bu hücre 1a'daki Ex 6'ya göre kaç satır tuttu? Aracın sana ne kazandırdığını bir cümleyle yaz.

> ...

### Araç 12 — `combine`
`combine(a, b, op)`: `op` → `"and"`, `"or"`, `"xor"`, `"minus"` (`a` ve değil-`b`). Bilinmeyen `op` → `ValueError`. Girdiler ve çıktı 0/255 `uint8` maske.

In [ ]:
import importlib, cvtools; importlib.reload(cvtools)
from cvtools import combine

a = np.array([[0, 0, 255, 255]], np.uint8)
b = np.array([[0, 255, 0, 255]], np.uint8)
expected = {
    "and":   [0, 0, 0, 255],
    "or":    [0, 255, 255, 255],
    "xor":   [0, 255, 255, 0],
    "minus": [0, 0, 255, 0],
}
for op, exp in expected.items():
    got = combine(a, b, op)
    assert got.dtype == np.uint8 and got.tolist() == [exp], f"{op}: beklenen {exp}, gelen {got.tolist()}"
try:
    combine(a, b, "nand")
    raise AssertionError("bilinmeyen op için ValueError bekleniyordu")
except ValueError:
    pass
print("Problem 12 ✓")

# Final Problem v2 — Cephe pencereleri, temizlenmiş

**Girdi:** 1a'daki aynı `images/facade.jpg` ve v1 karar kaydın.

**Çıktı:**
1. `outputs/facade_v2.png`: v2 kutuları
2. `outputs/windows/`: her pencere adayı ayrı bir dosya (`window_00.png`, `window_01.png`, ...). Bu senin ilk küçük veri setin.
3. v1 ve v2'nin yan yana karşılaştırması ve sayılar

**Kural:** Yine yeni görüntü işleme kodu yok; v1 zincirine yalnızca bu modülün araçlarını ekle.

```text
v1 zinciri: load_bgr → diagnose_exposure → (maske) → smooth → binarize
                                                                  │
v2 ekleri:                                    combine(..., "minus") ← gökyüzü/ağaç maskesi
                                                                  │
                                                             clean_mask   → karar: kernel boyutu ve şekli
                                                                  │
                                                             find_boxes → w/h filtresi → draw_boxes
                                                                  │
                                                             crop_boxes   → karar: pad kaç?
```

v1'de "hangi araç yetmedi?" sorusuna yazdığın cevaba bak: v2 o soruyu çözdü mü?

In [ ]:
import importlib, cvtools; importlib.reload(cvtools)
from cvtools import *

facade = load_bgr("images/facade.jpg")

# v1 zincirini buraya taşı, v2 eklerini ekle.
# Kesitleri kaydetmek için:
# out_dir = Path("outputs/windows"); out_dir.mkdir(parents=True, exist_ok=True)
# for i, p in enumerate(crop_boxes(facade, boxes, pad=...)):
#     cv2.imwrite(str(out_dir / f"window_{i:02d}.png"), p)

### Karar kaydı (v2)

| Adım | v1 | v2 | Neden değişti |
|---|---|---|---|
| maske birleştirme | — | | |
| clean_mask (open_k, close_k, kernel şekli) | — | | |
| min_area | | | |
| w/h filtresi | | | |
| crop pad | — | | |

**Sonuç:** v1 bulunan __ · v2 bulunan __ · gerçek __

**v2'nin hâlâ çözemediği:**

> ...

## Modül 1b kapanış kontrolü

- [ ] Ex 26–35 çalışıyor, her birinin altında gözlem var
- [ ] Rotation'da çıktı boyutu ve resize'da oran yorumlandı
- [ ] Problem 9–12 test hücreleri `✓`
- [ ] Final v2: `outputs/facade_v2.png`, `outputs/windows/` ve karar kaydı
- [ ] *Restart & Run All* hatasız